# 04 – Exploratory Data Analysis (EVISION 360)

Warehouse (`evision` schema, PostgreSQL) එකෙන් SQL වලින් data ගෙන global EV market එක explore කරනවා.

| Section | Business question |
| --- | --- |
| 1 | Ember data කොච්චර complete ද? (global totals විශ්වාස කරන්න පුළුවන්ද?) |
| 2 | Global EV market වර්ධනය වෙන්නේ කොහොමද? |
| 3 | BEV සහ PHEV අතර preference එක |
| 4 | ඉදිරියෙන් ඉන්න රටවල් මොනවාද? |
| 5 | Drivetrain mix (රටවල් අනුව) |
| 6 | Charging infrastructure |
| 7 | Economy සහ adoption අතර සම්බන්ධය |
| 8 | Outliers |
| 9 | Sri Lanka lens: සමාන ආර්ථික තත්ත්වයක ඉන්න රටවල් |
| 10 | Findings (data වලින්ම ගණනය කරපු) |

**සටහන:** correlation කියන්නේ causation නෙවෙයි. IEA projections (STEPS/CPS) scenarios මිස forecasts නෙවෙයි.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import text

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src/ingestion"))
from db import get_engine

engine = get_engine()
FIG = ROOT / "reports/figures"
FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 60)

# "Midnight Electric" palette (same direction as the planned dashboard)
BG, PANEL, TXT, GRID = "#0b1426", "#0f1d36", "#e6f1ff", "#22345a"
CYAN, GREEN, AMBER, PINK, GREY = "#00e5ff", "#39ff88", "#ffb347", "#ff5ea8", "#8fa3c7"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": PANEL, "savefig.facecolor": BG, "text.color": TXT,
    "axes.labelcolor": TXT, "axes.edgecolor": GRID, "xtick.color": TXT, "ytick.color": TXT,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.alpha": 0.6, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "legend.facecolor": PANEL, "legend.edgecolor": GRID,
})


def q(sql, params=None):
    """Run SQL against the evision schema and return a DataFrame."""
    with engine.connect() as conn:
        conn.exec_driver_sql("SET search_path TO evision")
        return pd.read_sql(text(sql), conn, params=params)


def annual(parameter, mode, powertrain, scenarios=("Historical",), geo_type="country",
           table="fact_ev_market_annual", keep_suspect=False):
    """IEA annual indicator as a DataFrame: geo_code, geo_name, year, scenario, value."""
    sql = f"""
        SELECT g.geo_code, g.geo_name, f.year, s.scenario_code AS scenario, f.value
        FROM {table} f
        JOIN dim_geography g USING (geo_key)
        JOIN dim_indicator i USING (indicator_key)
        JOIN dim_scenario s USING (scenario_key)
        WHERE i.parameter_code = :p AND i.vehicle_mode = :m AND i.powertrain = :pt
          AND s.scenario_code = ANY(:sc) AND g.geo_type = :gt
          {"" if keep_suspect else "AND NOT f.is_suspect_value"}"""
    return q(sql, {"p": parameter, "m": mode, "pt": powertrain, "sc": list(scenarios), "gt": geo_type})


def save(fig, name):
    fig.savefig(FIG / name, dpi=150, bbox_inches="tight")
    print("saved", name)


print(q("SELECT COUNT(*) AS geographies FROM dim_geography").iloc[0, 0], "geographies in the warehouse")

## 1. Data coverage: Ember global totals

Ember වල රටවල් ගාණ අවුරුදු අතර වෙනස් වෙනවා (සමහර රටවල් data පටන් ගන්නේ 2023 සිට). ඒ නිසා Ember total එක කෙලින්ම සසඳලා "growth" කියලා කියන්න බැහැ. IEA **World** (cars, BEV+PHEV) එක්ක සසඳලා coverage බලමු.

In [ ]:
ember = q("""
    SELECT d.year, COUNT(DISTINCT g.geo_key) AS ember_countries, SUM(f.registrations) AS ember_ev
    FROM fact_ev_sales_monthly f
    JOIN dim_geography g USING (geo_key)
    JOIN dim_drivetrain dr USING (drivetrain_key)
    JOIN dim_date d USING (month_start)
    WHERE g.geo_type = 'country' AND dr.drivetrain_code = 'EV_TOTAL' AND d.year BETWEEN 2015 AND 2025
    GROUP BY d.year ORDER BY d.year""")
world = annual("ev_sales", "Cars", "EV", geo_type="aggregate")
world = world[world["geo_code"] == "WLD"][["year", "value"]].rename(columns={"value": "iea_world_ev"})
cov = ember.merge(world, on="year")
cov["ember_coverage_pct"] = (cov["ember_ev"] / cov["iea_world_ev"] * 100).round(1)
print(cov.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(cov["year"], cov["ember_ev"] / 1e6, color=CYAN, label="Ember (countries reporting)")
ax.plot(cov["year"], cov["iea_world_ev"] / 1e6, color=GREEN, marker="o", label="IEA World (cars)")
ax.set_xticks(cov["year"])
ax.set_xticklabels([f"{y}{chr(10)}n={n}" for y, n in zip(cov["year"], cov["ember_countries"])], fontsize=8)
ax.set(title="Ember coverage grows over time (n = countries reporting)", ylabel="EV registrations (millions)", xlabel="")
ax.legend(); save(fig, "01_ember_coverage.png"); plt.show()

**Insight:** Ember වල coverage 2019 සිට IEA World එකෙන් ≈92–98%, ඒත් 2015–2018 වලදී 66–86% පමණයි. ඒ නිසා **global growth rate සඳහා IEA World** භාවිතා කරන්න, Ember භාවිතා කරන්නේ monthly pattern, country-level සහ drivetrain analysis වලටයි.

## 2. Global EV market growth (IEA World, cars)

In [ ]:
w = annual("ev_sales", "Cars", "EV", scenarios=("Historical", "STEPS", "CPS"), geo_type="aggregate")
w = w[w["geo_code"] == "WLD"]
hist = w[(w["scenario"] == "Historical") & w["year"].between(2015, 2025)].sort_values("year").copy()
hist["yoy_pct"] = hist["value"].pct_change() * 100
proj = w[(w["scenario"] != "Historical") & (w["year"] == 2035)]
print(hist[["year", "value", "yoy_pct"]].round(1).to_string(index=False))
print(proj[["scenario", "year", "value"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(hist["year"], hist["value"] / 1e6, color=CYAN)
for _, r in hist.iterrows():
    if pd.notna(r["yoy_pct"]):
        ax.text(r["year"], r["value"] / 1e6 + 0.6, f"{r['yoy_pct']:+.0f}%", ha="center", fontsize=8, color=GREEN)
for i, (_, r) in enumerate(proj.sort_values("scenario").iterrows()):
    ax.bar(2027.7 + i * 1.2, r["value"] / 1e6, width=1.1, color=AMBER if r["scenario"] == "CPS" else PINK,
           alpha=0.85, hatch="//", edgecolor=BG, label=f"2035 {r['scenario']} (scenario)")
ax.set(title="EV car sales: history (bars, YoY labels) and IEA 2035 scenarios (hatched, not forecasts)",
       ylabel="EV car sales (millions)")
ax.axvline(2026.9, color=GRID, ls="--", lw=1)
ax.set_xticks(list(range(2015, 2026)) + [2028.3])
ax.set_xticklabels([str(y) for y in range(2015, 2026)] + ["2035" + chr(10) + "scenarios"], fontsize=9)
ax.legend(); save(fig, "02_global_ev_sales.png"); plt.show()

## 3. BEV vs PHEV

In [ ]:
bev = annual("ev_sales", "Cars", "BEV", geo_type="aggregate").query("geo_code == 'WLD'").set_index("year")["value"]
phev = annual("ev_sales", "Cars", "PHEV", geo_type="aggregate").query("geo_code == 'WLD'").set_index("year")["value"]
mix = pd.DataFrame({"BEV": bev, "PHEV": phev}).loc[2015:2025]
mix["bev_share_pct"] = mix["BEV"] / (mix["BEV"] + mix["PHEV"]) * 100
print(mix.round(1).to_string())

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(mix.index, mix["BEV"] / 1e6, color=CYAN, label="BEV")
ax.bar(mix.index, mix["PHEV"] / 1e6, bottom=mix["BEV"] / 1e6, color=GREEN, label="PHEV")
ax.set(title="World EV car sales: BEV vs PHEV (IEA)", ylabel="millions"); ax.legend()
save(fig, "03_bev_vs_phev_world.png"); plt.show()

# Countries where PHEV outsells BEV (2024, markets above 10k EV cars)
b = annual("ev_sales", "Cars", "BEV").query("year == 2024").set_index("geo_name")["value"]
p = annual("ev_sales", "Cars", "PHEV").query("year == 2024").set_index("geo_name")["value"]
cmp_ = pd.DataFrame({"BEV": b, "PHEV": p}).dropna()
cmp_ = cmp_[(cmp_["BEV"] + cmp_["PHEV"]) >= 10_000]
cmp_["phev_share_pct"] = cmp_["PHEV"] / (cmp_["BEV"] + cmp_["PHEV"]) * 100
print("\nPHEV-heavy markets 2024:"); print(cmp_.sort_values("phev_share_pct", ascending=False).head(8).round(0).to_string())

## 4. Country comparison (2024, IEA, cars)

In [ ]:
sales = annual("ev_sales", "Cars", "EV").query("year == 2024")[["geo_code", "geo_name", "value"]].rename(columns={"value": "ev_sales"})
share = annual("ev_sales_share", "Cars", "EV").query("year == 2024")[["geo_code", "value"]].rename(columns={"value": "ev_share_pct"})
c24 = sales.merge(share, on="geo_code")
top_share = c24[c24["ev_sales"] >= 20_000].nlargest(12, "ev_share_pct").iloc[::-1]
top_vol = c24.nlargest(12, "ev_sales").iloc[::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].barh(top_share["geo_name"], top_share["ev_share_pct"], color=GREEN)
axes[0].set(title="EV share of new car sales, 2024 (markets with 20k+ EV sales)", xlabel="%")
axes[1].barh(top_vol["geo_name"], top_vol["ev_sales"] / 1e6, color=CYAN)
axes[1].set(title="EV car sales volume, 2024", xlabel="millions")
plt.tight_layout(); save(fig, "04_country_ranking_2024.png"); plt.show()
print("Highest share overall (any volume):"); print(c24.nlargest(5, "ev_share_pct").round(1).to_string(index=False))

**සටහන:** Share ranking එකට කුඩා markets (Nepal, Iceland වැනි) ඇතුළත් වුණාම වෙනස් වෙනවා. ඒ නිසා volume filter එකක් (20k+) දාලා ඉහළ ranking එක පෙන්වනවා. IEA values round කරලා තියෙන්නේ (උදා 92%, 74%).

## 5. Drivetrain mix (Ember, 2024, largest markets)

In [ ]:
mixq = q("""
    SELECT g.geo_name, dr.drivetrain_code, SUM(f.registrations) AS regs
    FROM fact_ev_sales_monthly f
    JOIN dim_geography g USING (geo_key)
    JOIN dim_drivetrain dr USING (drivetrain_key)
    JOIN dim_date d USING (month_start)
    WHERE g.geo_type = 'country' AND d.year = 2024 AND NOT dr.is_aggregate
    GROUP BY 1, 2""")
allq = q("""
    SELECT g.geo_name, SUM(f.registrations) AS all_regs, COUNT(DISTINCT f.month_start) AS months
    FROM fact_ev_sales_monthly f
    JOIN dim_geography g USING (geo_key)
    JOIN dim_drivetrain dr USING (drivetrain_key)
    JOIN dim_date d USING (month_start)
    WHERE g.geo_type = 'country' AND d.year = 2024 AND dr.drivetrain_code = 'ALL'
    GROUP BY 1""")
big = allq[allq["months"] == 12].nlargest(8, "all_regs")["geo_name"]
comp = mixq[mixq["geo_name"].isin(big)].pivot(index="geo_name", columns="drivetrain_code", values="regs").fillna(0)
share_comp = comp.div(comp.sum(axis=1), axis=0) * 100
order = [c for c in ["BEV", "PHEV", "HEV", "PETROL", "DIESEL", "ETHANOL_BLEND", "ICE_OTHER", "OTHER"] if c in share_comp]
share_comp = share_comp[order]
share_comp = share_comp.loc[share_comp[["BEV", "PHEV"]].sum(axis=1).sort_values().index]  # highest EV share on top
colors = {"BEV": CYAN, "PHEV": GREEN, "HEV": AMBER, "PETROL": GREY, "DIESEL": "#5b6f94", "ETHANOL_BLEND": PINK,
          "ICE_OTHER": "#3a4a6b", "OTHER": "#2a3550"}
fig, ax = plt.subplots(figsize=(10, 4.8))
left = np.zeros(len(share_comp))
for col in share_comp.columns:
    ax.barh(share_comp.index, share_comp[col], left=left, color=colors[col], label=col)
    left += share_comp[col].values
ax.set(title="New-car drivetrain mix, 2024 (reported drivetrains only)", xlabel="% of registrations"); ax.legend(ncol=8, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.14))
save(fig, "05_drivetrain_mix_2024.png"); plt.show()
print(share_comp.round(1).to_string())

**සටහන:** රටවල් සියල්ලම සියලු drivetrains report කරන්නේ නැහැ. මේ chart එක තියෙන්නේ report කරපු drivetrains අතර බෙදීම පමණයි.

## 6. Charging infrastructure (2024)

In [ ]:
cp = {}
for label, pt in {"fast": "Publicly available fast", "slow": "Publicly available slow", "ultra_fast": "Publicly available ultra-fast"}.items():
    cp[label] = annual("charging_points", "EVSE", pt, table="fact_charging_infrastructure").query("year == 2024").set_index("geo_name")["value"]
cps = pd.DataFrame(cp)
both = cps.dropna(subset=["fast", "ultra_fast"])
n_ultra_gt = int((both["ultra_fast"] > both["fast"]).sum())
print(f"Countries where ultra-fast points exceed fast points: {n_ultra_gt} of {len(both)}")
print("-> ultra-fast cannot be a subset of fast, so the three categories look separate (still confirm in the IEA documentation).\n")
cps["total_public"] = cps[["fast", "slow", "ultra_fast"]].sum(axis=1)
top_cp = cps.nlargest(10, "total_public").iloc[::-1]
print((top_cp / 1000).round(1).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(top_cp)); h = 0.27
ax.barh(y + h, top_cp["slow"].fillna(0) / 1e3, h, color=GREY, label="slow")
ax.barh(y, top_cp["fast"].fillna(0) / 1e3, h, color=CYAN, label="fast")
ax.barh(y - h, top_cp["ultra_fast"].fillna(0) / 1e3, h, color=GREEN, label="ultra-fast")
ax.set_yticks(y); ax.set_yticklabels(top_cp.index)
ax.set(title="Publicly available charging points, 2024 (thousands)", xlabel="thousand charging points"); ax.legend()
save(fig, "06a_charging_points_2024.png"); plt.show()

**සටහන:** සමහර රටවල් වල (උදා: Germany, USA) ultra-fast points, fast points වලට වඩා වැඩියි. ඒ නිසා ultra-fast කියන්නේ fast වල subset එකක් නෙවෙයි, categories තුන වෙන් වෙලා තියෙනවා කියලා සැලකිය හැකියි. ඒත් IEA documentation එකෙන් definitions තහවුරු කරගන්න. `total_public` එක ඒ උපකල්පනය මත රඳා පවතිනවා.

In [ ]:
epc = annual("ev_per_charging_point", "EV", "EV", table="fact_charging_infrastructure").query("year == 2024")[["geo_code", "geo_name", "value"]].rename(columns={"value": "ev_per_charger"})
sc = epc.merge(share, on="geo_code").merge(sales[["geo_code", "ev_sales"]], on="geo_code")
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(sc["ev_share_pct"], sc["ev_per_charger"], s=30 + sc["ev_sales"] / sc["ev_sales"].max() * 400, color=CYAN, alpha=0.7)
for _, r in sc.nlargest(8, "ev_sales").iterrows():
    ax.annotate(r["geo_name"], (r["ev_share_pct"], r["ev_per_charger"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_yscale("log")
ax.set(title="EVs per public charging point vs EV sales share, 2024 (bubble = EV sales)",
       xlabel="EV share of new car sales (%)", ylabel="EVs per charging point (log scale)")
save(fig, "06b_evs_per_charger.png"); plt.show()
print("EVs per charging point, median:", round(sc["ev_per_charger"].median(), 1), "| countries:", len(sc))

**සටහන:** EVs per charging point කියන්නේ charging availability එකේ සම්පූර්ණ මිනුමක් නෙවෙයි (home charging, charger power, භූගෝලීය විසිරීම ඇතුළත් නැහැ).

## 7. Economy vs adoption (2024)

In [ ]:
econ = q("""
    SELECT g.geo_code, g.geo_name, e.gdp_per_capita_usd, e.urban_pop_pct, e.population
    FROM fact_economic_indicators e JOIN dim_geography g USING (geo_key)
    WHERE g.geo_type = 'country' AND e.year = 2024""")
ea = c24.merge(econ.drop(columns="geo_name"), on="geo_code")
print("Countries with both EV share and economic data:", len(ea))

fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.scatter(ea["gdp_per_capita_usd"], ea["ev_share_pct"], color=GREEN, alpha=0.75,
           s=30 + ea["ev_sales"] / ea["ev_sales"].max() * 400)
for _, r in ea.nlargest(7, "ev_sales").iterrows():
    ax.annotate(r["geo_name"], (r["gdp_per_capita_usd"], r["ev_share_pct"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
for name in ["Norway", "Nepal"]:
    r = ea[ea["geo_name"] == name]
    if len(r):
        ax.annotate(name, (r["gdp_per_capita_usd"].iloc[0], r["ev_share_pct"].iloc[0]), fontsize=8, xytext=(4, -10), textcoords="offset points")
ax.set_xscale("log")
lka_gdp = econ.loc[econ["geo_code"] == "LKA", "gdp_per_capita_usd"].iloc[0]
ax.axvline(lka_gdp, color=AMBER, ls="--", lw=1.2)
ax.text(lka_gdp * 1.05, 85, "Sri Lanka" + chr(10) + "GDP per capita", color=AMBER, fontsize=8)
ax.set(title="GDP per capita vs EV share of new car sales, 2024", xlabel="GDP per capita (US$, log scale)", ylabel="EV share (%)")
save(fig, "07_gdp_vs_ev_share.png"); plt.show()

ea["log_population"] = np.log10(ea["population"])
corr = ea[["ev_share_pct", "gdp_per_capita_usd", "urban_pop_pct", "log_population"]].corr(method="spearman")["ev_share_pct"].drop("ev_share_pct")
print("Spearman rank correlation with EV share:"); print(corr.round(2).to_string())

**සටහන:** Spearman rank correlation (outliers සඳහා ශක්තිමත්). මේක සම්බන්ධතාවක් (association) මිස හේතුවක් නෙවෙයි. Policy, ඉන්ධන මිල, vehicle supply වැනි සාධක ඇතුළත් නැහැ. Norway (92%) සහ Nepal (74%) වැනි outliers GDP එකෙන් පැහැදිලි වෙන්නේ නැහැ.

## 8. Outliers

In [ ]:
print("Flagged impossible values (kept in warehouse, excluded above):")
print(q("""
    SELECT g.geo_name, i.parameter_code, i.vehicle_mode, f.year, f.value
    FROM fact_ev_market_annual f JOIN dim_geography g USING (geo_key) JOIN dim_indicator i USING (indicator_key)
    WHERE f.is_suspect_value ORDER BY 1, 4""").to_string(index=False))

m = q("""
    SELECT g.geo_name, f.month_start, f.registrations
    FROM fact_ev_sales_monthly f JOIN dim_geography g USING (geo_key) JOIN dim_drivetrain dr USING (drivetrain_key)
    WHERE g.geo_type = 'country' AND dr.drivetrain_code = 'EV_TOTAL' AND f.is_complete_period
    ORDER BY 1, 2""")
MIN_BASE = 200  # ignore tiny volumes: a jump from 0 to 3 registrations is not a meaningful spike
rows = []
for name, g in m.groupby("geo_name"):
    s = g.set_index("month_start")["registrations"].astype(float)
    if len(s) < 24 or s.median() < 500:
        continue
    base = s.rolling(5, center=True, min_periods=3).median()
    ratio = s / base
    flagged = ratio[(base >= MIN_BASE) & ((ratio > 2.5) | (ratio < 0.4))]
    for d, r in flagged.items():
        rows.append({"country": name, "month": d, "registrations": int(s[d]), "local_median": int(base[d]), "ratio": round(r, 2)})
spikes = pd.DataFrame(rows)
print(f"\nMonthly spikes/drops (vs 5-month rolling median), {len(spikes)} flagged:")
print(spikes.sort_values("ratio", ascending=False).head(10).to_string(index=False) if len(spikes) else "none")
print("Flagged months by calendar month:", pd.to_datetime(spikes["month"]).dt.month.value_counts().sort_index().to_dict() if len(spikes) else {})

පරීක්ෂාව: මාසික EV registrations 5-මාස rolling median එකෙන් 2.5× ට වැඩි හෝ 0.4× ට අඩු, ඒ වගේම ඒ median එක 200 ට වැඩි මාස පමණක් flag කරනවා. ගොඩක් flags **December සහ January** මාස වලට වැටෙනවා (උඩ calendar-month ගණන බලන්න). ඒකට හේතු (year-end push, tax හෝ policy වෙනස්කම්) මම තහවුරු කරගෙන නැහැ. ඒ නිසා මේවා errors කියලා ගන්න එපා, අයින් කරන්නත් එපා, **investigate කරන්න** flag කරපු ඒවා විදිහට සලකන්න.

## 9. Sri Lanka lens: සමාන ආර්ථික තත්ත්වයක ඉන්න රටවල්

Sri Lanka ට EV data නැහැ (Ember, IEA දෙකෙහිම නැහැ). ඒත් World Bank data තියෙනවා. GDP per capita (2024) එකෙන් **0.5× – 2×** අතර EV data ඇති රටවල් පෙන්වමු.

In [ ]:
lka = econ[econ["geo_code"] == "LKA"].iloc[0]
lo, hi = lka["gdp_per_capita_usd"] * 0.5, lka["gdp_per_capita_usd"] * 2
peers = ea[ea["gdp_per_capita_usd"].between(lo, hi)].merge(epc[["geo_code", "ev_per_charger"]], on="geo_code", how="left")
peers = peers[["geo_name", "gdp_per_capita_usd", "urban_pop_pct", "ev_sales", "ev_share_pct", "ev_per_charger"]]
print(f"Sri Lanka 2024: GDP per capita ${lka['gdp_per_capita_usd']:,.0f}, urban {lka['urban_pop_pct']:.1f}%")
print(f"Peers (GDP per capita ${lo:,.0f} - ${hi:,.0f}), {len(peers)} countries:")
print(peers.sort_values("ev_share_pct", ascending=False).round(1).to_string(index=False))

**සටහන:** මේක benchmark එකක් පමණයි, Sri Lanka ගේ EV adoption ගැන predict කිරීමක් නෙවෙයි. ආනයන ප්‍රතිපත්ති, ඉන්ධන මිල, charging network වැනි Sri Lanka-specific සාධක ඇතුළත් නැහැ. Readiness Index එකේදී (Phase 07) මේක structural සාධක වලින් නිසි ලෙස විශ්ලේෂණය කරමු.

## 10. Findings (data වලින්ම ගණනය කරපු)

In [ ]:
c2018 = cov.loc[cov["year"] == 2018, "ember_coverage_pct"].iloc[0]
c2025 = cov.loc[cov["year"] == 2025, "ember_coverage_pct"].iloc[0]
v2020 = hist.loc[hist["year"] == 2020, "value"].iloc[0]; v2025 = hist.loc[hist["year"] == 2025, "value"].iloc[0]
steps35 = proj.loc[proj["scenario"] == "STEPS", "value"].iloc[0]; cps35 = proj.loc[proj["scenario"] == "CPS", "value"].iloc[0]
lead = top_share.iloc[-1]; vol = top_vol.iloc[-1]
print(f"1. Ember covers ~{c2018:.0f}% of IEA's world EV sales in 2018 but ~{c2025:.0f}% in 2025, so use IEA for global growth.")
print(f"2. World EV car sales: {v2020/1e6:.1f}M (2020) -> {v2025/1e6:.1f}M (2025), x{v2025/v2020:.1f}.")
print(f"3. IEA 2035 scenarios: STEPS {steps35/1e6:.0f}M vs CPS {cps35/1e6:.0f}M EV cars (scenarios, not forecasts).")
print(f"4. BEV share of world EV car sales in 2025: {mix.loc[2025, 'bev_share_pct']:.0f}% ({mix.loc[2015, 'bev_share_pct']:.0f}% in 2015).")
print(f"5. Highest EV share (20k+ EV sales): {lead['geo_name']} {lead['ev_share_pct']:.0f}%. Largest market: {vol['geo_name']} ({vol['ev_sales']/1e6:.1f}M).")
print(f"6. Spearman rho with EV share (2024, n={len(ea)}): GDP per capita {corr['gdp_per_capita_usd']:.2f}, urban share {corr['urban_pop_pct']:.2f}.")
print(f"7. {len(spikes)} monthly spikes/drops flagged for review; {len(peers)} countries have a GDP per capita within 0.5x-2x of Sri Lanka's.")